In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
import emlearn
import os

# 1. LOAD AND CLEAN DATA
print("Loading dataset...")
df = pd.read_csv('household_power_consumption.csv', sep=',', low_memory=False)

# Take 50,000 rows for faster training
df = df.head(50000)

# Clean missing data (marked as '?')
df['Global_active_power'] = pd.to_numeric(df['Global_active_power'], errors='coerce')
df = df.dropna(subset=['Global_active_power'])
data = df['Global_active_power'].values

# 2. CREATE SLIDING WINDOW (temporal sequences)
window_size = 5
X = []
y = []

for i in range(len(data) - window_size):
    X.append(data[i : i + window_size])
    y.append(data[i + window_size])

X = np.array(X)
y = np.array(y)

# 3. SPLIT INTO TRAINING AND TEST
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, shuffle=True)

# 4. FEATURE NORMALIZATION
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('\n=== TO COPY INTO sensor.c ===')
print(f'static const float FEATURE_MEAN[{window_size}] = {{ {", ".join([f"{m:.4f}f" for m in scaler.mean_])} }};')
print(f'static const float FEATURE_SCALE[{window_size}] = {{ {", ".join([f"{s:.4f}f" for s in scaler.scale_])} }};')
print('=========================================\n')

# 5. BUILD AND TRAIN NEURAL NETWORK
print("Building TensorFlow model...")
model = tf.keras.Sequential([
    # First hidden layer: 5 input values -> 16 neurons
    tf.keras.layers.Dense(16, activation='relu', input_shape=(window_size,)),
    # Second hidden layer: 16 neurons -> 16 neurons
    tf.keras.layers.Dense(16, activation='relu'),
    # Output layer: 1 neuron for power prediction (regression)
    tf.keras.layers.Dense(1) 
])

# Compile: MSE and MAE metrics
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

# Early stopping: monitor validation loss and restore best weights
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

print("\nTraining in progress...")
history = model.fit(
    x=X_train_scaled,
    y=y_train,
    epochs=100,
    validation_data=(X_test_scaled, y_test),
    callbacks=[early_stop],
    shuffle=True,
)

# 6. EVALUATE MODEL PERFORMANCE
print("\n=== Model Evaluation ===")
test_loss, test_mae = model.evaluate(X_test_scaled, y_test, batch_size=32)
print(f'Test MSE (Mean Squared Error): {test_loss:.4f}')
print(f'\nTest MAE (Mean Absolute Error): {test_mae:.4f} kW')
print('========================\n')

# 7. CONVERT MODEL FOR IOT DEVICE WITH EMLEARN
path = 'power_predictor.h'
print("Converting model to C code with emlearn...")
cmodel = emlearn.convert(model, method='inline')
cmodel.save(file=path, name='power_predictor')

print(f'Model exported to: {path}')

# 8. VERIFY RAM USAGE (nRF52840 has 256KB)
size_bytes = os.path.getsize(path)
nrf52840_ram = 256 * 1024
print(f'\nModel file size: {size_bytes / 1024:.1f} KB')
print(f'Device RAM: {nrf52840_ram // 1024} KB')
print(f'Model occupies: {size_bytes / nrf52840_ram * 100:.1f}% of device RAM')